In [1]:
!pip install pandas openpyxl pymysql sqlalchemy

In [15]:
#'''
#项目名称：多数据源电商订单自动对账系统
#功能概述：
#1.数据源：MySQL / Excel / CSV 多渠道混合接入
#2.能力：订单合并、缺失渠道检测、金额不一致识别
#3.报表：带时间戳归档Excel，双sheet全量表+差异表；失败自动生成故障报告
#4.日志：全程运行日志持久化保存
#适用场景：电商多平台流水、代理商台账、多方业务数据勾对
#'''
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine

In [16]:

#=======================【配置文件 + 底层工具函数（数据接入层）：支持MySQL+Excel混合数据源】=======================
#- 全局路径、渠道配置字典
#- write_log 日志函数
#- load_excel_bill / load_csv_bill / load_mysql_bill
#- load_all_bills 统一调度加载、容错跳过失效渠道

LOG_FILE = "../unified_bill_log.txt"
OUTPUT_EXCEL = "对账输出报告.xlsx"
# 在列表里为每个渠道单独指定数据源 source="mysql" / source="excel"
TARGET_CHANNEL_CONFIG = [
    {"name":"合作分销商台账","source":"excel"},
    {"name":"平台后台账单","source":"csv"}   
]
#TARGET_CHANNEL_CONFIG = [
#    {"name":"合作分销商台账","source":"excel"},
#    {"name":"ERP内部数据","source":"mysql"},
#    {"name":"平台后台账单","source":"csv"}
#]
#TARGET_CHANNEL_CONFIG = [
#    {"name":"ERP内部数据","source":"mysql"},
#    {"name":"平台后台账单","source":"csv"}
#]
#=====================================================================
COL_ORDER = "order_id"
COL_AMOUNT = "amount"
COL_TIME = "create_time"

# ----------------------MySQL数据库配置----------------------
DB_CONFIG = {
    "user":"root",
    "password":"你的数据库密码",
    "host":"127.0.0.1",
    "port":3306,
    "db":"ecommerce_db"
}
#===============================================================================

# ----------------------Excel渠道与文件名映射----------------------
EXCEL_FILE_MAP = {
    "合作分销商台账":"../data/internal.xlsx"    
}

# ----------------------CSV渠道与文件名映射（source="csv"生效）新增----------------------
CSV_FILE_MAP = {
    "平台后台账单":"../data/channelA.csv"
}
#=========================================================================================

#日志函数【修复：增加encoding="utf-8"】
def write_log(content:str):
    now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_line = f"[{now_str}] {content}\n"
    with open(LOG_FILE,"a",encoding="utf-8") as f:
        f.write(log_line)
    print(log_line.strip())

#加载Excel单渠道账单
def load_excel_bill(file_path,channel_name):
    df = pd.read_excel(
        file_path,
        dtype={COL_ORDER:str},
        usecols=[COL_ORDER,COL_AMOUNT]
    )
    df = df.dropna(subset=[COL_ORDER])
    df[COL_ORDER] = df[COL_ORDER].str.strip()
    df = df.rename(columns={COL_AMOUNT:f"金额_{channel_name}"})
    return df

#【新增】加载CSV单渠道账单：多编码识别 + 金额清洗
def load_csv_bill(file_path,channel_name):
    #1、自动识别编码：utf-8-sig -> utf-8 -> gbk
    for enc in ["utf-8-sig","utf-8","gbk"]:
        try:
            df = pd.read_csv(
                file_path,
                encoding=enc,
                dtype={COL_ORDER:str},
                usecols=[COL_ORDER,COL_AMOUNT]
            )
            break
        except (UnicodeDecodeError, ValueError):
            continue
    else:
        raise RuntimeError(f"CSV[{file_path}] 编码无法识别，请确认为utf-8或gbk")
    #2、金额列清洗：去除 ¥ ￥ 逗号 空格，统一转数值
    df[COL_AMOUNT] = pd.to_numeric(
        df[COL_AMOUNT].astype(str).str.replace(r"[¥￥,\s]","",regex=True),
        errors="coerce"
    )
    #3、脏数据过滤
    df = df.dropna(subset=[COL_ORDER])
    df[COL_ORDER] = df[COL_ORDER].str.strip()
    df = df.rename(columns={COL_AMOUNT:f"金额_{channel_name}"})
    return df

#加载MySQL统一单表的单个渠道账单【修复完成版】
def load_mysql_bill(channel_name):
    try:
        sql = """
            SELECT order_id,amount,create_time
            FROM all_channel_bill
            WHERE channel_name = %s
        """
        conn_str = f'mysql+pymysql://{DB_CONFIG["user"]}:{DB_CONFIG["password"]}@{DB_CONFIG["host"]}:{DB_CONFIG["port"]}/{DB_CONFIG["db"]}?charset=utf8mb4'
        engine = create_engine(conn_str)
        df = pd.read_sql(sql,engine,params=[channel_name],dtype={COL_ORDER:str})
        df = df.dropna(subset=[COL_ORDER])
        df[COL_ORDER] = df[COL_ORDER].str.strip()
        df = df.rename(columns={COL_AMOUNT:f"金额_{channel_name}"})
        return df
    except Exception:
        err_msg = f"【MySQL加载失败】渠道[{channel_name}]，数据库无法连接/查询失败！本机未安装MySQL或账号密码错误，请切换为Excel数据源！"
        write_log(err_msg)
        return None

#逐个渠道按自身数据源读取，支持混合
def load_all_bills():
    bill_dict = {}
    for item in TARGET_CHANNEL_CONFIG:
        ch_name = item["name"]
        src_type = item["source"]
        try:
            if src_type == "mysql":
                write_log(f"正在从MySQL加载渠道【{ch_name}】")
                df = load_mysql_bill(ch_name)
            elif src_type == "excel":
                write_log(f"正在从Excel加载渠道【{ch_name}】")
                file_path = EXCEL_FILE_MAP[ch_name]
                df = load_excel_bill(file_path,ch_name)
            elif src_type == "csv":
                write_log(f"正在从CSV加载渠道【{ch_name}】")
                df = load_csv_bill(CSV_FILE_MAP[ch_name], ch_name)
            else:
                write_log(f"渠道[{ch_name}]配置source非法，仅支持 mysql / excel，跳过该渠道")
                continue
            #关键修复：只有df不为None，才加入结果字典
            if df is not None:
                bill_dict[ch_name]=df
                write_log(f"{ch_name} 读取完成，订单数：{len(df)}")
            else:
                write_log(f"渠道【{ch_name}】加载返回空数据，跳过")
        except Exception:
            write_log(f"渠道【{ch_name}】加载失败，跳过此渠道！")
    return bill_dict


In [17]:
#=====================【对账核心引擎（业务算法层，独立模块）】=======================
#单独一层，和文件读取、报表输出解耦，突出你的数据处理能力
#注释说明：外连接合并多渠道、异常判定规则、差额计算逻辑

def check_universal_channel(df_list,channel_names):
    full_df = df_list[0]
    for d in df_list[1:]:
        full_df = pd.merge(full_df, d, on=COL_ORDER, how="outer")
    channel_cnt=len(channel_names)
    def judge(row):
        amt_dict = {}
        for c in channel_names:
            col = f"金额_{c}"
            val = row[col]
            if pd.notna(val):
                amt_dict[c]=round(val,2)
        exist_channels = list(amt_dict.keys())
        exist_count = len(exist_channels)
        total_channels = channel_cnt
        if exist_count < total_channels:
            missing = list(set(channel_names)-set(exist_channels))
            return f"缺失渠道：{','.join(missing)}"
        amounts = list(amt_dict.values())
        first_amt=amounts[0]
        all_same = all(abs(x-first_amt)<0.01 for x in amounts)
        if not all_same:
            return "各渠道金额不一致"
        return ""
    full_df["异常说明"]=full_df.apply(judge,axis=1)
    #双渠道增加差额列，增强可读性
    if channel_cnt ==2:
        c1,c2 = channel_names[0],channel_names[1]
        full_df["差额"]=full_df[f"金额_{c1}"] - full_df[f"金额_{c2}"]
    diff_result = full_df[full_df["异常说明"]!=""].copy()
    return full_df,diff_result


In [18]:
#==============【主调度入口（应用层，程序启动入口）】================
#- 统一调度加载
#- 判断可用渠道数量
#- 分支：正常对账 / 对账失败
#- 生成时间戳 Excel、多 sheet 报表、日志输出、页面展示结果

import os

if __name__ == "__main__":
    write_log("=======启动自动化对账任务【支持MySQL+Excel+CSV混合数据源】=======")
    bill_data = load_all_bills()
    channel_names = list(bill_data.keys())
    df_list = list(bill_data.values())
    channel_cnt = len(channel_names)

    #自动创建report文件夹
    report_dir = "../output_sample/output_sample"
    if not os.path.exists(report_dir):
        os.mkdir(report_dir)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    #文件输出到report目录
    OUTPUT_EXCEL = f"{report_dir}/对账报告_{timestamp}.xlsx"

    if channel_cnt <2:
        msg = f"对账执行失败，成功加载的有效渠道数量：{channel_cnt}，至少需要2个渠道才能对账，请检查数据源配置！"
        write_log(msg)
        fail_df = pd.DataFrame({"执行结果":[msg]})
        with pd.ExcelWriter(OUTPUT_EXCEL,engine="openpyxl") as writer:
            fail_df.to_excel(writer,sheet_name="执行失败说明",index=False)
    else:
        full_df,diff_df = check_universal_channel(df_list,channel_names)
        diff_count = len(diff_df)
        write_log(f"本次检测差异订单总数：{diff_count}")
        with pd.ExcelWriter(OUTPUT_EXCEL,engine="openpyxl") as writer:
            full_df.to_excel(writer,sheet_name="全部订单总表",index=False)
            diff_df.to_excel(writer,sheet_name="差异明细表",index=False)
        write_log(f"对账报告已生成：{OUTPUT_EXCEL}，含【全部订单总表】+【差异明细表】\n")
        display(full_df)
        display(diff_df)

[2026-09-12 12:16:14] =======启动自动化对账任务【支持MySQL+Excel+CSV混合数据源】=======
[2026-09-12 12:16:14] 正在从Excel加载渠道【合作分销商台账】
[2026-09-12 12:16:14] 合作分销商台账 读取完成，订单数：217
[2026-09-12 12:16:14] 正在从CSV加载渠道【平台后台账单】
[2026-09-12 12:16:14] 平台后台账单 读取完成，订单数：218
[2026-09-12 12:16:14] 本次检测差异订单总数：13
[2026-09-12 12:16:14] 对账报告已生成：../output_sample/output_sample/对账报告_20260912_121614.xlsx，含【全部订单总表】+【差异明细表】


,order_id,金额_合作分销商台账,金额_平台后台账单,异常说明,差额
0,'1942876092019358792,54.00,54.00,,0.0
1,'1942954392295289367,84.90,84.90,,0.0
2,'1943053068391598282,54.00,54.00,,0.0
3,'1943411700070901579,52.38,52.38,,0.0
4,'1943411808031744095,54.00,54.00,,0.0
...,...,...,...,...,...
218,'3471059702341608639,NaN,140.09,缺失渠道：合作分销商台账,NaN
219,'3471318470740478115,49.00,49.00,,0.0
220,'3471720842973990736,49.00,188.00,各渠道金额不一致,-139.0
221,'3471737475757490241,134.65,134.65,,0.0


,order_id,金额_合作分销商台账,金额_平台后台账单,异常说明,差额
55,'1948180548568160489,348.50,NaN,缺失渠道：平台后台账单,NaN
66,'1948572734911479368,NaN,146.55,缺失渠道：合作分销商台账,NaN
69,'1948842661380248277,NaN,112.01,缺失渠道：合作分销商台账,NaN
177,'3468488653198642611,1119.61,NaN,缺失渠道：平台后台账单,NaN
188,'3468879973295899509,114.27,NaN,缺失渠道：平台后台账单,NaN
199,'3469516563171719139,231.67,NaN,缺失渠道：平台后台账单,NaN
203,'3469920588778672561,NaN,69.84,缺失渠道：合作分销商台账,NaN
207,'3470232241086957350,182.84,1182.84,各渠道金额不一致,-1000.0
209,'3470278141572465059,NaN,98.68,缺失渠道：合作分销商台账,NaN
211,'3470393090914778361,NaN,80.11,缺失渠道：合作分销商台账,NaN


In [20]:
!pip freeze > ../requirements.txt